# Introduction

This case study is part of the Google Data Analytics Professional Certification. The objective is to leverage a data set from a fictional company called Bellabeat to answer a series of key business questions. In order to answer the key business questions, I will follow the steps of the data analysis process: Ask, Prepare, Process, Analyze, Share, and Act.

## Bellabeat

Bellabeat is a high-tech manufacturer of health-focused products for women. Bellabeat is a successful small company, but they have the potential to become a larger player in the global smart device market. Urška Sršen, cofounder and Chief Creative Officer of Bellabeat, believes that analyzing smart device fitness data could help unlock new growth opportunities for the company.

# 1. Ask

## 1.1 Goal

The primary objective is to analyze smart device usage data from non-Bellabeat consumers (FitBit Fitness Tracker Data) to identify global trends in health and wellness tracking. By understanding how people interact with their smart devices, the goal is to apply these insights to a specific Bellabeat product and provide high-level, data-driven recommendations to improve the company’s marketing strategy and unlock new growth opportunities in the global market

## 1.2 Stakeholders

• Urška Sršen: Bellabeat’s cofounder and Chief Creative Officer; primary requester of the analysis.

• Sando Mur: Mathematician and Bellabeat’s cofounder; a key member of the executive team.

• Bellabeat Marketing Analytics Team: The team of data analysts responsible for guiding business goals through data.

## 1.3 Scope

Analyze smart device usage to gain insights into how consumers use non-FitBit smart devices to inform a new marketing strategy.

# 2. Prepare

## 2.1 Data Integrity

The data's quality is assessed using the ROCCC framework (Reliable, Original, Comprehensive, Current, and Cited). While useful for identifying habits, the dataset has limitations regarding sample size (only 30 users), potential bias, is not current (quite old), and the timeframe is at most 31 days for each user, with one user having data only for 4 days (not Comprehensive).

## 2.2 Privacy & Ethics

The data is de-identified and all users are assigned a randomized integer.

## 2.3 Storage

The original data is pulled from Kaggle. It will then be stored between two subfolders: /raw and /processed.

# 3. Process


## 3.1 Importing Libraries

The datasets will be pulled from Kaggle and stored locally using the kagglehub and shutil packages. This method will ensure that if the datasets are updated in the future, these datasets will pull the newest data. I am also using Polars package. Polars is a new, high-performance, open-source library for data manipulation and analysis. It offers significant performance gains for large datasets over libraries like Pandas because its core engine is written in Rust. While these datasets are not large by any means, the exposure to Polars is still warranted.

In [1]:
import kagglehub
import shutil
import polars as pl
import os
import re

C:\Users\jacob\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3.2 Loading Datasets

The FitBit datasets are saved to a "raw_data" folder within the project folder. With the datasets downloaded and placed into the raw_data folder, three of the CSVs will be read in: dailyActivity_merged, sleepDay_merged, and weightLogInfo_merged.

In [2]:
dataset_identifier = "arashnic/fitbit"
download_path = "C:/Users/jacob/repositories/python/bellabeat_case_study/raw_data"
cache_path = kagglehub.dataset_download(dataset_identifier)
os.makedirs(download_path, exist_ok=True)
shutil.copytree(cache_path, download_path, dirs_exist_ok=True)

print("Path to dataset files:", download_path)

Path to dataset files: C:/Users/jacob/repositories/python/bellabeat_case_study/raw_data


In [3]:
files = os.listdir(download_path + "/mturkfitbit_export_4.12.16-5.12.16/Fitabase Data 4.12.16-5.12.16")
print("Files found:", files)

Files found: ['dailyActivity_merged.csv', 'dailyCalories_merged.csv', 'dailyIntensities_merged.csv', 'dailySteps_merged.csv', 'heartrate_seconds_merged.csv', 'hourlyCalories_merged.csv', 'hourlyIntensities_merged.csv', 'hourlySteps_merged.csv', 'minuteCaloriesNarrow_merged.csv', 'minuteCaloriesWide_merged.csv', 'minuteIntensitiesNarrow_merged.csv', 'minuteIntensitiesWide_merged.csv', 'minuteMETsNarrow_merged.csv', 'minuteSleep_merged.csv', 'minuteStepsNarrow_merged.csv', 'minuteStepsWide_merged.csv', 'sleepDay_merged.csv', 'weightLogInfo_merged.csv']


In [4]:
path = download_path + "/mturkfitbit_export_4.12.16-5.12.16/Fitabase Data 4.12.16-5.12.16"

activity_file = os.path.join(path, 'dailyActivity_merged.csv')
sleep_file = os.path.join(path, 'sleepDay_merged.csv')
weight_file = os.path.join(path, 'weightLogInfo_merged.csv')

activity_df = pl.read_csv(activity_file, infer_schema_length=1000)
sleep_df = pl.read_csv(sleep_file, infer_schema_length=1000)
weight_df = pl.read_csv(weight_file, infer_schema_length=1000)

In [5]:
print(activity_df.shape)
print(activity_df.head())
print(activity_df.schema)

(940, 15)
shape: (5, 15)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Id        ┆ ActivityD ┆ TotalStep ┆ TotalDist ┆ … ┆ FairlyAct ┆ LightlyAc ┆ Sedentary ┆ Calories │
│ ---       ┆ ate       ┆ s         ┆ ance      ┆   ┆ iveMinute ┆ tiveMinut ┆ Minutes   ┆ ---      │
│ i64       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ s         ┆ es        ┆ ---       ┆ i64      │
│           ┆ str       ┆ i64       ┆ f64       ┆   ┆ ---       ┆ ---       ┆ i64       ┆          │
│           ┆           ┆           ┆           ┆   ┆ i64       ┆ i64       ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 150396036 ┆ 4/12/2016 ┆ 13162     ┆ 8.5       ┆ … ┆ 13        ┆ 328       ┆ 728       ┆ 1985     │
│ 6         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ 150396036 ┆ 4/13/2016 ┆ 10735     ┆ 6.97      ┆ … ┆ 19        ┆ 

In [6]:
print(sleep_df.shape)
print(sleep_df.head())
print(sleep_df.schema)

(413, 5)
shape: (5, 5)
┌────────────┬───────────────────────┬───────────────────┬────────────────────┬────────────────┐
│ Id         ┆ SleepDay              ┆ TotalSleepRecords ┆ TotalMinutesAsleep ┆ TotalTimeInBed │
│ ---        ┆ ---                   ┆ ---               ┆ ---                ┆ ---            │
│ i64        ┆ str                   ┆ i64               ┆ i64                ┆ i64            │
╞════════════╪═══════════════════════╪═══════════════════╪════════════════════╪════════════════╡
│ 1503960366 ┆ 4/12/2016 12:00:00 AM ┆ 1                 ┆ 327                ┆ 346            │
│ 1503960366 ┆ 4/13/2016 12:00:00 AM ┆ 2                 ┆ 384                ┆ 407            │
│ 1503960366 ┆ 4/15/2016 12:00:00 AM ┆ 1                 ┆ 412                ┆ 442            │
│ 1503960366 ┆ 4/16/2016 12:00:00 AM ┆ 2                 ┆ 340                ┆ 367            │
│ 1503960366 ┆ 4/17/2016 12:00:00 AM ┆ 1                 ┆ 700                ┆ 712            │
└──────

In [7]:
print(weight_df.shape)
print(weight_df.head())
print(weight_df.schema)

(67, 8)
shape: (5, 8)
┌────────────┬─────────────┬───────────┬─────────────┬──────┬───────────┬─────────────┬────────────┐
│ Id         ┆ Date        ┆ WeightKg  ┆ WeightPound ┆ Fat  ┆ BMI       ┆ IsManualRep ┆ LogId      │
│ ---        ┆ ---         ┆ ---       ┆ s           ┆ ---  ┆ ---       ┆ ort         ┆ ---        │
│ i64        ┆ str         ┆ f64       ┆ ---         ┆ i64  ┆ f64       ┆ ---         ┆ i64        │
│            ┆             ┆           ┆ f64         ┆      ┆           ┆ bool        ┆            │
╞════════════╪═════════════╪═══════════╪═════════════╪══════╪═══════════╪═════════════╪════════════╡
│ 1503960366 ┆ 5/2/2016    ┆ 52.599998 ┆ 115.963147  ┆ 22   ┆ 22.65     ┆ true        ┆ 1462233599 │
│            ┆ 11:59:59 PM ┆           ┆             ┆      ┆           ┆             ┆ 000        │
│ 1503960366 ┆ 5/3/2016    ┆ 52.599998 ┆ 115.963147  ┆ null ┆ 22.65     ┆ true        ┆ 1462319999 │
│            ┆ 11:59:59 PM ┆           ┆             ┆      ┆        

## 3.3 Data Cleaning

This next section will be to clean the datasets to prepare them for analysis.

### 3.3.1 Column Names

We saw earlier that the column names are in PascalCase, so to keep with industry standards we will convert them to snake_case. Since the column names are in PascalCase, it will take using Regex to get all of the column names into snake_case.

In [8]:
def to_snake_case (name: str) -> str:
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    s2 = re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower()
    return s2.replace(" ", "_").replace("-","_")

In [9]:
activity_df, sleep_df, weight_df = [df.rename(to_snake_case) for df in [activity_df, sleep_df, weight_df]]

print(activity_df.schema)
print(sleep_df.schema)
print(weight_df.schema)

Schema({'id': Int64, 'activity_date': String, 'total_steps': Int64, 'total_distance': Float64, 'tracker_distance': Float64, 'logged_activities_distance': Float64, 'very_active_distance': Float64, 'moderately_active_distance': Float64, 'light_active_distance': Float64, 'sedentary_active_distance': Float64, 'very_active_minutes': Int64, 'fairly_active_minutes': Int64, 'lightly_active_minutes': Int64, 'sedentary_minutes': Int64, 'calories': Int64})
Schema({'id': Int64, 'sleep_day': String, 'total_sleep_records': Int64, 'total_minutes_asleep': Int64, 'total_time_in_bed': Int64})
Schema({'id': Int64, 'date': String, 'weight_kg': Float64, 'weight_pounds': Float64, 'fat': Int64, 'bmi': Float64, 'is_manual_report': Boolean, 'log_id': Int64})


### 3.3.2 Date Column

With the column names handled, we realize that the three datasets all use a different name for their date columns and they are strings. So we will make all three datasets have the same name for their 'date' column and convert them from string to dates.

In [10]:
activity_df = activity_df.rename({'activity_date': 'date'})
sleep_df = sleep_df.rename({'sleep_day': 'date'})

activity_df = activity_df.with_columns(pl.col("date").str.to_date("%m/%d/%Y"))
sleep_df = sleep_df.with_columns(pl.col("date").str.split(" ").list.get(0).str.to_date("%m/%d/%Y"))
weight_df = weight_df.with_columns(pl.col("date").str.split(" ").list.get(0).str.to_date("%m/%d/%Y"))

print(activity_df.head())
print(sleep_df.head())
print(weight_df.head())

shape: (5, 15)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ id        ┆ date      ┆ total_ste ┆ total_dis ┆ … ┆ fairly_ac ┆ lightly_a ┆ sedentary ┆ calories │
│ ---       ┆ ---       ┆ ps        ┆ tance     ┆   ┆ tive_minu ┆ ctive_min ┆ _minutes  ┆ ---      │
│ i64       ┆ date      ┆ ---       ┆ ---       ┆   ┆ tes       ┆ utes      ┆ ---       ┆ i64      │
│           ┆           ┆ i64       ┆ f64       ┆   ┆ ---       ┆ ---       ┆ i64       ┆          │
│           ┆           ┆           ┆           ┆   ┆ i64       ┆ i64       ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 150396036 ┆ 2016-04-1 ┆ 13162     ┆ 8.5       ┆ … ┆ 13        ┆ 328       ┆ 728       ┆ 1985     │
│ 6         ┆ 2         ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ 150396036 ┆ 2016-04-1 ┆ 10735     ┆ 6.97      ┆ … ┆ 19        ┆ 217       

### 3.3.3 Missing Data

Next, we check for missing values. The only dataset that is missing values is the weight_df. In this case, however, we cannot simply drop the rows with missing data because the missing data is all from a single column, 'fat', and it is missing data for 65 out of 67 rows. With only two rows having data for this column, we will just drop the column as a whole.

In [11]:
print(activity_df.null_count(), sleep_df.null_count(), weight_df.null_count())

shape: (1, 15)
┌─────┬──────┬─────────────┬──────────────┬───┬─────────────┬─────────────┬─────────────┬──────────┐
│ id  ┆ date ┆ total_steps ┆ total_distan ┆ … ┆ fairly_acti ┆ lightly_act ┆ sedentary_m ┆ calories │
│ --- ┆ ---  ┆ ---         ┆ ce           ┆   ┆ ve_minutes  ┆ ive_minutes ┆ inutes      ┆ ---      │
│ u32 ┆ u32  ┆ u32         ┆ ---          ┆   ┆ ---         ┆ ---         ┆ ---         ┆ u32      │
│     ┆      ┆             ┆ u32          ┆   ┆ u32         ┆ u32         ┆ u32         ┆          │
╞═════╪══════╪═════════════╪══════════════╪═══╪═════════════╪═════════════╪═════════════╪══════════╡
│ 0   ┆ 0    ┆ 0           ┆ 0            ┆ … ┆ 0           ┆ 0           ┆ 0           ┆ 0        │
└─────┴──────┴─────────────┴──────────────┴───┴─────────────┴─────────────┴─────────────┴──────────┘ shape: (1, 5)
┌─────┬──────┬─────────────────────┬──────────────────────┬───────────────────┐
│ id  ┆ date ┆ total_sleep_records ┆ total_minutes_asleep ┆ total_time_in_bed │
│ -

In [12]:
weight_df = weight_df.drop("fat")
print(weight_df.head())

shape: (5, 7)
┌────────────┬────────────┬───────────┬───────────────┬───────────┬────────────────┬───────────────┐
│ id         ┆ date       ┆ weight_kg ┆ weight_pounds ┆ bmi       ┆ is_manual_repo ┆ log_id        │
│ ---        ┆ ---        ┆ ---       ┆ ---           ┆ ---       ┆ rt             ┆ ---           │
│ i64        ┆ date       ┆ f64       ┆ f64           ┆ f64       ┆ ---            ┆ i64           │
│            ┆            ┆           ┆               ┆           ┆ bool           ┆               │
╞════════════╪════════════╪═══════════╪═══════════════╪═══════════╪════════════════╪═══════════════╡
│ 1503960366 ┆ 2016-05-02 ┆ 52.599998 ┆ 115.963147    ┆ 22.65     ┆ true           ┆ 1462233599000 │
│ 1503960366 ┆ 2016-05-03 ┆ 52.599998 ┆ 115.963147    ┆ 22.65     ┆ true           ┆ 1462319999000 │
│ 1927972279 ┆ 2016-04-13 ┆ 133.5     ┆ 294.31712     ┆ 47.540001 ┆ false          ┆ 1460509732000 │
│ 2873212765 ┆ 2016-04-21 ┆ 56.700001 ┆ 125.002104    ┆ 21.450001 ┆ true     

### 3.3.4 Duplicate Data

Lastly, we check for any duplicated data. We are only concerned with rows that are complete duplicates (i.e. two or more rows that are identical across all columns). We find the sleep_df has three sets of complete duplicates, so we drop the duplicates.

In [13]:
print(activity_df.filter(activity_df.is_duplicated()), sleep_df.filter(sleep_df.is_duplicated()), weight_df.filter(weight_df.is_duplicated()))

shape: (0, 15)
┌─────┬──────┬─────────────┬──────────────┬───┬─────────────┬─────────────┬─────────────┬──────────┐
│ id  ┆ date ┆ total_steps ┆ total_distan ┆ … ┆ fairly_acti ┆ lightly_act ┆ sedentary_m ┆ calories │
│ --- ┆ ---  ┆ ---         ┆ ce           ┆   ┆ ve_minutes  ┆ ive_minutes ┆ inutes      ┆ ---      │
│ i64 ┆ date ┆ i64         ┆ ---          ┆   ┆ ---         ┆ ---         ┆ ---         ┆ i64      │
│     ┆      ┆             ┆ f64          ┆   ┆ i64         ┆ i64         ┆ i64         ┆          │
╞═════╪══════╪═════════════╪══════════════╪═══╪═════════════╪═════════════╪═════════════╪══════════╡
└─────┴──────┴─────────────┴──────────────┴───┴─────────────┴─────────────┴─────────────┴──────────┘ shape: (6, 5)
┌────────────┬────────────┬─────────────────────┬──────────────────────┬───────────────────┐
│ id         ┆ date       ┆ total_sleep_records ┆ total_minutes_asleep ┆ total_time_in_bed │
│ ---        ┆ ---        ┆ ---                 ┆ ---                  ┆ ---  

In [14]:
sleep_df = sleep_df.unique()
print(sleep_df.filter(sleep_df.is_duplicated()))

shape: (0, 5)
┌─────┬──────┬─────────────────────┬──────────────────────┬───────────────────┐
│ id  ┆ date ┆ total_sleep_records ┆ total_minutes_asleep ┆ total_time_in_bed │
│ --- ┆ ---  ┆ ---                 ┆ ---                  ┆ ---               │
│ i64 ┆ date ┆ i64                 ┆ i64                  ┆ i64               │
╞═════╪══════╪═════════════════════╪══════════════════════╪═══════════════════╡
└─────┴──────┴─────────────────────┴──────────────────────┴───────────────────┘


### 3.3.5 Export Cleaned Data

With the datasets cleaned, we export the datasets to a clean_data folder to keep them separate from the raw_data.

In [15]:
clean_data_path = "C:/Users/jacob/repositories/python/bellabeat_case_study/clean_data"
os.makedirs(clean_data_path, exist_ok=True)

activity_df.write_csv(clean_data_path + "/daily_activity_clean.csv")
sleep_df.write_csv(clean_data_path + "/daily_sleep_clean.csv")
weight_df.write_csv(clean_data_path + "/weight_log_clean.csv")

# 4. Analyze

In [16]:
daily_activity_clean = pl.read_csv("daily_activity_cleaned.csv")
daily_activity_clean.describe()

statistic,id,activity_date,total_steps,total_distance,tracker_distance,logged_activities_distance,very_active_distance,moderately_active_distance,light_active_distance,sedentary_active_distance,very_active_minutes,fairly_active_minutes,lightly_active_minutes,sedentary_minutes,calories
str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",940.0,"""940""",940.0,940.0,940.0,940.0,940.0,940.0,940.0,940.0,940.0,940.0,940.0,940.0,940.0
"""null_count""",0.0,"""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",4.8554e9,null,7637.910638,5.489702,5.475351,0.108171,1.502681,0.567543,3.340819,0.001606,21.164894,13.564894,192.812766,991.210638,2303.609574
"""std""",2.4248e9,null,5087.150742,3.924606,3.907276,0.619897,2.658941,0.88358,2.040655,0.007346,32.844803,19.987404,109.1747,301.267437,718.166862
"""min""",1.5040e9,"""2016-04-12""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",2.3201e9,null,3790.0,2.62,2.62,0.0,0.0,0.0,1.95,0.0,0.0,0.0,127.0,730.0,1829.0
"""50%""",4.4451e9,null,7412.0,5.25,5.25,0.0,0.21,0.24,3.37,0.0,4.0,6.0,199.0,1058.0,2135.0
"""75%""",6.9622e9,null,10725.0,7.71,7.71,0.0,2.04,0.8,4.78,0.0,32.0,19.0,264.0,1229.0,2793.0
"""max""",8.8777e9,"""2016-05-12""",36019.0,28.030001,28.030001,4.942142,21.92,6.48,10.71,0.11,210.0,143.0,518.0,1440.0,4900.0
